# Week 6

Finally made the classes done :))

## Phi vals

In [15]:
from src.api.v4 import HMM, GaussEmission, StaticTransition, AutoregressiveGaussEmission
import matplotlib.pyplot as plt
import jax.numpy as jnp 

## VALUES FROM TEXTBOOK EXAMPLE
mean = jnp.array([400, 536, 890, 1307]) 
sd = jnp.array([22, 77, 111, 197])  
u0 = jnp.array([0.19, 0.35, 0.15, 0.32])
Gamma = jnp.array([
            [8.5e-01, 1.5e-01, 0.0028, 9.7e-09],
            [8.3e-02, 8.5e-01, 0.0679, 6.8e-09],
            [4.3e-04, 1.6e-01, 0.6842, 1.5e-01],
            [2.6e-08, 6.1e-09, 0.0723, 9.3e-01],]
            )

gauss_emission = GaussEmission.from_params(mu=mean, sigma=sd)
arr_emission = AutoregressiveGaussEmission.from_params(mu=mean, sigma=sd, phi=jnp.array([0.2, 0.4, 0.8, 0.5]))
transition = StaticTransition.from_params(Gamma)

hmm = HMM(transition=transition, emission=gauss_emission, inital_distribution=u0)
ar_hmm = HMM(transition=transition, emission=arr_emission, inital_distribution=u0) 

In [16]:
from src.api.v4.utils import load_y_data

ys = load_y_data() 

In [17]:
from src.api.v4 import LBFGSSolver


hmm.fit(ys=ys, solver=LBFGSSolver(), frozen={"mu0": False})
ar_hmm.fit(ys=ys, solver=LBFGSSolver(), frozen={"mu0": False})


In [ ]:
print(f"Phi vals for AR HMM: {ar_hmm.emission.phi()}")
print(f"Phi tilde is: {ar_hmm.emission.phi_tilde}")

print(ar_hmm.emission.mu(10, ys, None)) 

Phi vals for AR HMM: [[0.25800243 0.44656616 0.8969112  0.73618734]]
Phi tilde is: (Array([-1.056377  , -0.21455453,  2.1633658 ,  1.0262454 ], dtype=float32),)
[406.69193 466.3037  480.13623 698.8367 ]


## Noter

1. Sæt de to laveste mu værdier til at være ens. 
2. 

In [ ]:
mu0 = ar_hmm.emission.mu0 
log_mu_diff = ar_hmm.emission.log_mu_diff


Mu vals: [5.7805774e+02 6.9623219e+04 2.8952938e+07]


In [19]:
import equinox as eqx
from src.api.v4.emissions.autoregressive_gauss_emission import phi_to_phi_tilde

phi0_vals = jnp.linspace(0.25, 0.30, num=10)
log_likelihoods = []

for phi0 in phi0_vals:
    # Immutably set lag-0, state-0 phi_tilde to logit(phi0)
    new_phi_tilde_0 = ar_hmm.emission.phi_tilde[0].at[0].set(phi_to_phi_tilde(phi0))
    new_emission = eqx.tree_at(lambda e: e.phi_tilde[0], ar_hmm.emission, new_phi_tilde_0)
    ar_hmm.params = eqx.tree_at(lambda p: p.emission, ar_hmm.params, new_emission)

    ar_hmm.fit(ys=ys, solver=LBFGSSolver(), frozen={"phi_tilde": {0: False}})
    print(f"Updated phi0 to {phi0:.2f}, now evaluating log likelihood...") 
    print(ar_hmm.emission.phi())
    ll = ar_hmm.log_likelihood(ys)
    log_likelihoods.append(ll)
    print(f"Phi0: {phi0:.2f}, Log Likelihood: {ll:.2f}")

Updated phi0 to 0.25, now evaluating log likelihood...
[[0.25       0.44656616 0.8969112  0.73618734]]
Phi0: 0.25, Log Likelihood: -9530.24
Updated phi0 to 0.26, now evaluating log likelihood...
[[0.25555554 0.44656616 0.8969112  0.73618734]]
Phi0: 0.26, Log Likelihood: -9530.68
Updated phi0 to 0.26, now evaluating log likelihood...
[[0.26111114 0.44656616 0.8969112  0.73618734]]
Phi0: 0.26, Log Likelihood: -9531.70
Updated phi0 to 0.27, now evaluating log likelihood...
[[0.26666665 0.44656616 0.8969112  0.73618734]]
Phi0: 0.27, Log Likelihood: -9533.25
Updated phi0 to 0.27, now evaluating log likelihood...
[[0.27222222 0.44656616 0.8969112  0.73618734]]
Phi0: 0.27, Log Likelihood: -9535.19
Updated phi0 to 0.28, now evaluating log likelihood...
[[0.2777778  0.44656616 0.8969112  0.73618734]]
Phi0: 0.28, Log Likelihood: -9534.42
Updated phi0 to 0.28, now evaluating log likelihood...
[[0.28333333 0.44656616 0.8969112  0.73618734]]
Phi0: 0.28, Log Likelihood: -9533.69
Updated phi0 to 0.29